In [0]:
%sql
drop table if exists instagram.silverlayer.silver_instagram_users;
drop table if exists instagram.silverlayer.quarantine_instagram_users;


In [0]:
# Importing and creating Spark Sesssion
from pyspark.sql import  SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("Production_ETL") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

In [0]:
# Loading the delta tables 
instagram_df = spark.read.table("instagram.bronzelayer.instagram_usage_lifestyle")
print(instagram_df.printSchema())
# instagram_df.show()
print("Columns in Instagram DF : ", instagram_df.columns)

null_counts=instagram_df.select([
    F.count(F.when(F.col(column).isNull(),column)).alias(column)
    for column in instagram_df.columns
])
null_counts.show()

In [0]:
# Gets Latest User Data
window_spec=Window.partitionBy("user_id").orderBy(F.col("last_login_date").desc())
instagram_df=instagram_df.withColumn("row_num",F.row_number().over(window_spec))\
    .filter(F.col("row_num")==1).drop("row_num")


In [0]:
# function to replace null with zero
def replace_null_with_zero(df,columns):
    for column in columns:
        df = df.withColumn(
            column, 
            F.coalesce(F.col(column), F.lit(0))
            )
    return df

# function to replace negative
def clean_numeric_data(df,columns):
    for column in columns:
        df=df.withColumn(
            column,
             F.when(F.col(column) < 0, 0).otherwise(F.coalesce(F.col(column), F.lit(0)))
             )
    return df

# Checks and add constarin for critical columns  
def add_violation(df, condition, message):
    return df.withColumn("rejection_reason", 
        F.when(condition, 
               F.when(F.col("rejection_reason") == "", F.lit(message))
                .otherwise(F.concat(F.col("rejection_reason"), F.lit(" | "), F.lit(message)))
        ).otherwise(F.col("rejection_reason"))
    )


# Filtering numeric Column
numeric_types=['int','bigint','double','float','decimal']
numeric_columns = [c for c, t in instagram_df.dtypes if any(x in t for x in numeric_types)]

# Filtering String Column
string_types=["string"]
string_columns=[
    column for column,dtype in instagram_df.dtypes 
    if  dtype  =="string"
]
    
# Cleaning : 1.Remove NA values 2.Replace negative values with zero 3.Normalize String Columns
# instagram_df =instagram_df.dropna(subset=string_columns)
instagram_df = clean_numeric_data(instagram_df,numeric_columns)
for col_name in string_columns:
    instagram_df = instagram_df.withColumn(col_name, F.initcap(F.trim(F.col(col_name))))

# filtering critical NA columns
instagram_df=instagram_df.withColumn("rejection_reason",F.lit(""))
for c in ["user_id", "app_name", "country"]:
    instagram_df = add_violation(instagram_df, F.col(c).isNull(), f"Null in {c}")


In [0]:
gender=instagram_df.select("gender").distinct()
income_lvl=instagram_df.select("income_level").distinct()
diet=instagram_df.select("diet_quality").distinct()
urban=instagram_df.select("urban_rural").distinct()
print("Gender :",gender.show())
print("Income Level :",income_lvl.show())
print("Diet Quality :",diet.show())
print("Urban Rural :",urban.show())



In [0]:
# Validating low cardinality columns
VALID_GENDERS=["male","female","non-binary","prefer not to say"]
VALID_INCOME=["lower-middle","middle","high","low","upper-middle"]
VALID_URBAN = ["urban", "rural","suburban"]
DIET_QUALITY=["very poor","poor","average","good","very good","excellent"]

# Validating Marrital Status
is_unmarried_with_kids = (F.lower(F.col("relationship_status")) == "unmarried") & (F.col("has_children") == "Yes")
is_underage_married = (F.lower(F.col("relationship_status")) == "married") & (F.col("age") < 18)

# Basic checks
is_invalid_age = (F.col("age") < 12) | (F.col("age") >= 100)
is_missing_id = F.col("user_id").isNull()
is_invalid_income = F.col("income_level").isNull()

# Physical Integrity Check
is_impossible_hours = (F.col("exercise_hours_per_week") > 168) | (F.col("sleep_hours_per_night") > 24) 
is_future_login = (F.col("last_login_date") > F.current_date())
is_broken_engagement = (F.col("user_engagement_score") < 0) | (F.col("user_engagement_score") > 10)

# Insta constrain check
is_time_on_reel_and_daily_active_min_check=F.col("time_on_reels_per_day") > F.col("daily_active_minutes_instagram")


# Create the flagging column
instagram_df = instagram_df.withColumn("rejection_reason", 
    F.when(is_missing_id, "Missing User ID")
     .when(is_invalid_age, "Age Out of Bounds")
     .when(is_unmarried_with_kids, "Logic Conflict: Unmarried with Kids")
     .when(is_underage_married, "Logic Conflict: Underage Married")
     .when(is_invalid_income, "Invalid Income Level")
     .when(is_impossible_hours, "Impossible Hours")
     .when(is_future_login, "Future Login")
     .when(is_broken_engagement, "Broken Engagement")
     .when(is_time_on_reel_and_daily_active_min_check, "Logic Conflict: Reels > Daily Active")
     .when(~F.lower(F.col("income_level")).isin(VALID_INCOME), "Invalid Income Level")
     .when(~F.lower(F.col("urban_rural")).isin(VALID_URBAN), "Invalid Urban/Rural Value")
     .when(~F.lower(F.col("gender")).isin(VALID_GENDERS), "Invalid Gender")
     .when(~F.lower(F.col("diet_quality")).isin(DIET_QUALITY), "Invalid Diet Quality")
     .otherwise("Valid")
)
# Add a simple boolean flag for quick filtering
instagram_df = instagram_df.withColumn("is_valid", F.col("rejection_reason") == "Valid")
instagram_df.columns


In [0]:
diet_map = {
    "Verypoor": "Very Poor", 
    "Verygood": "Very Good"
}

gender_map = {
    "Prefernottosay": "Prefer Not To Say"
}

instagram_df = instagram_df.withColumn(
    "diet_quality",
    F.coalesce(
        F.create_map([F.lit(x) for x in sum(diet_map.items(), ())])[F.col("diet_quality")],
        F.col("diet_quality")
    )
)

instagram_df = instagram_df.withColumn(
    "gender",
    F.coalesce(
        F.create_map([F.lit(x) for x in sum(gender_map.items(), ())])[F.col("gender")],
        F.col("gender")
    )
)

instagram_df = instagram_df.withColumn(
    "income_level",
    F.when(F.col("income_level") == "Lower-middle", "Lower-Middle")
     .when(F.col("income_level") == "Upper-middle", "Upper-Middle")
     .otherwise(F.col("income_level"))
)

silver_df = instagram_df.filter(F.col("is_valid") == True).drop("rejection_reason", "is_valid")
quarantine_df = instagram_df.filter(F.col("is_valid") == False)

silver_df.printSchema()

In [0]:
display(
    instagram_df.filter(F.col("rejection_reason") != "Valid")
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
silver_df = silver_df.withColumn("user_id", F.col("user_id").cast("int")) \
                     .withColumn("user_engagement_score", F.col("user_engagement_score").cast("double"))

total=instagram_df.count()
valid=silver_df.count()
invalid=total-valid
print("Total Records: ",total)
print("Valid Records: ",valid)
print("Invalid Records: ",invalid)
display(silver_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["user_id", "age", "income_level"]]))


In [0]:
# Func to insert,upsert in the silver layer
from delta.tables import DeltaTable
def save_or_upsert_silver(df, table_name, join_key="user_id"):
    if not spark.catalog.tableExists(table_name):
        print(f"🚀 Table {table_name} not found. Creating as new Delta table...")
        df.write.format("delta") \
          .mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(table_name)
    else:
        print(f"🔄 Table {table_name} exists. Performing upsert (Merge)...")
        target_table = DeltaTable.forName(spark, table_name)
        
        # Execute the Merge (SCD Type 1)
        (target_table.alias("target")
         .merge(
             df.alias("source"),
             f"target.{join_key} = source.{join_key}"
         )
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
        print(f"✅ Upsert complete for {table_name}")
# Save the Clean Data
save_or_upsert_silver(silver_df, "instagram.silverlayer.silver_instagram_users")

# Save the Quarantine for the Quality Team to fix
quarantine_df.write.format("delta").mode("append").saveAsTable("instagram.silverlayer.quarantine_instagram_users")